***
# ICAD Data Management
***

In [2]:
#Import libraries
import os, sys, json
import pandas as pd
import numpy as np
import toml

In [3]:
#Set Pandas options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [6]:
!pip install psycopg2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.7/385.7 kB 5.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for psycopg2: filename=psycopg2-2.9.10-cp312-cp312-macosx_11_0_arm64.whl size=133582 sha256=03c6456e4fe33a7f7c373d770f731e0938d3102ce9eea35b571ac2a43035e29d
  Stored in directory: /Users/prathampandey/Library/Caches/pip/wheels/ac/bb/ce/afa589c50b6004d3a06fc691e71bd09c9bd5f01e5921e5329b
Successfully built psycopg2


In [4]:
wdir = os.getcwd()
wdir

'/Users/prathampandey/Documents/Code/Aks/icad_data_management'

In [7]:
#Change directory to database operations to load all the methods
#Import all methods from database_operations
from database_operations.create_database_function import create_postgres_db
from database_operations.create_df_from_sql_function import create_df_from_sql
from database_operations.create_table_from_pandas_df_function import create_table_from_df

from data_cleaning.clean_column_names import clean_column_names
from data_cleaning.clean_strings import clean_string_data

from data_matching.get_master_school import get_standard_school

In [12]:
# Load config
config = toml.load(wdir + "/config.toml")
print (config)

{'postgres_credentials': {'username': 'youruser', 'password': 'yourpassword', 'host': '127.0.0.1', 'port': '5432'}, 'db_details': {'dbname': 'icad_db', 'school_master': 'school_master', 'all_students': 'all_students', 'compitative_exams': 'compitative_exams', 'cpa_results': 'cpa_results', 'exam_marks': 'exam_marks', 'admissions': 'admissions'}, 'file_locations': {'all_schools': '/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/activity_report.csv'}}


In [10]:
#Create database in postgres called icad_db
create_postgres_db(config['postgres_credentials'], config['db_details']['dbname'])

Error:  FATAL:  database "youruser" does not exist



UnboundLocalError: cannot access local variable 'connection' where it is not associated with a value


***
## Load all school data in a table
***

In [13]:
#Read all schools data from csv
all_schools_df = pd.read_csv(config["file_locations"]["all_schools"])

In [14]:
all_schools_df.head(10)

,institute_name,address1,address2,city,district,state,board,contact_no,email_address,category,branch_name
0,ABC Convent and High School,Manewada,Manewada,Nagpur,Nagpur,Maharashtra,State,9096300320,NaN,B,EAST CENTRE - NAGPUR
1,Adarsh Sanskar Vidyalaya Hasanbagh,Hasanbagh,Hasanbagh,Nagpur,Nagpur,Maharashtra,State,NaN,NaN,B,EAST CENTRE - NAGPUR
2,Adarsh Sanskar Vidyalaya CBSE Hudkeshwar,Hudkeshwar,Hudkeshwar,Nagpur,Nagpur,Maharashtra,State,7620264605,NaN,B,EAST CENTRE - NAGPUR
3,Adarsh Vidya Mandir,CA Road,CA Road,Nagpur,Nagpur,Maharashtra,State,NaN,NaN,B,EAST CENTRE - NAGPUR
4,Adarsh Vidyalaya,Umrer,Umrer,Umred,Nagpur,Maharashtra,State,NaN,NaN,B,EAST CENTRE - NAGPUR
5,Amit High School,Dighori,Dighori,Nagpur,Nagpur,Maharashtra,State,NaN,NaN,C,EAST CENTRE - NAGPUR
6,Ankush Classes,Near dada Saheb thakre High School ; Aashirwaad Nagar Hudkeshwar Sq,Aashirwaad Nagar,Nagpur,Nagpur,Maharashtra,State,9881015403,noid@emailid.com,B,EAST CENTRE - NAGPUR
7,Ashok Kanya Vidyalaya,Umrer,Umrer,Umred,Nagpur,Maharashtra,State,NaN,NaN,C,EAST CENTRE - NAGPUR
8,Ashok Vidyalaya Boys,Umrer,Umrer,Umred,Nagpur,Maharashtra,State,NaN,NaN,C,EAST CENTRE - NAGPUR
9,Baba Nanak Sidhi Hindi High School,Nandanwan,Nandanwan,Nagpur,Nagpur,Maharashtra,State,7122764288,babananaksindhihindihs@gmail.com,C,EAST CENTRE - NAGPUR


In [15]:
all_schools_df.dtypes

institute_name    object
 address1         object
 address2         object
 city             object
 district         object
 state            object
 board            object
 contact_no       object
 email_address    object
category          object
branch_name       object
dtype: object

In [16]:
all_schools_df.columns

Index(['institute_name', ' address1', ' address2', ' city', ' district',
       ' state', ' board', ' contact_no', ' email_address', 'category',
       'branch_name'],
      dtype='object')

In [17]:
#Clean column names
all_schools_df = clean_column_names(all_schools_df)

In [18]:
all_schools_df.columns

Index(['institute_name', 'address1', 'address2', 'city', 'district', 'state',
       'board', 'contact_no', 'email_address', 'category', 'branch_name'],
      dtype='object')

In [19]:
#Clean school names, addresses
all_schools_df['institute_name'] = all_schools_df['institute_name'].apply(lambda x: clean_string_data(x))
all_schools_df['address1'] = all_schools_df['address1'].apply(lambda x: clean_string_data(x))
all_schools_df['address2'] = all_schools_df['address2'].apply(lambda x: clean_string_data(x))
all_schools_df['address2'] = all_schools_df['address2'].apply(lambda x: clean_string_data(x))
all_schools_df['city'] = all_schools_df['city'].apply(lambda x: clean_string_data(x))
all_schools_df['state'] = all_schools_df['state'].apply(lambda x: clean_string_data(x))

In [20]:
all_schools_df.head()

,institute_name,address1,address2,city,district,state,board,contact_no,email_address,category,branch_name
0,ABC CONVENT AND HIGH SCHOOL,MANEWADA,MANEWADA,NAGPUR,Nagpur,MAHARASHTRA,State,9096300320,NaN,B,EAST CENTRE - NAGPUR
1,ADARSH SANSKAR VIDYALAYA HASANBAGH,HASANBAGH,HASANBAGH,NAGPUR,Nagpur,MAHARASHTRA,State,NaN,NaN,B,EAST CENTRE - NAGPUR
2,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR,HUDKESHWAR,HUDKESHWAR,NAGPUR,Nagpur,MAHARASHTRA,State,7620264605,NaN,B,EAST CENTRE - NAGPUR
3,ADARSH VIDYA MANDIR,CA ROAD,CA ROAD,NAGPUR,Nagpur,MAHARASHTRA,State,NaN,NaN,B,EAST CENTRE - NAGPUR
4,ADARSH VIDYALAYA,UMRER,UMRER,UMRED,Nagpur,MAHARASHTRA,State,NaN,NaN,B,EAST CENTRE - NAGPUR


In [21]:
# Define unique identifier for a school
all_schools_ids = all_schools_df.rename_axis('school_id').reset_index()
all_schools_ids['school_id'] = all_schools_ids['school_id'] + 1000

In [22]:
all_schools_ids.head()

,school_id,institute_name,address1,address2,city,district,state,board,contact_no,email_address,category,branch_name
0,1000,ABC CONVENT AND HIGH SCHOOL,MANEWADA,MANEWADA,NAGPUR,Nagpur,MAHARASHTRA,State,9096300320,NaN,B,EAST CENTRE - NAGPUR
1,1001,ADARSH SANSKAR VIDYALAYA HASANBAGH,HASANBAGH,HASANBAGH,NAGPUR,Nagpur,MAHARASHTRA,State,NaN,NaN,B,EAST CENTRE - NAGPUR
2,1002,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR,HUDKESHWAR,HUDKESHWAR,NAGPUR,Nagpur,MAHARASHTRA,State,7620264605,NaN,B,EAST CENTRE - NAGPUR
3,1003,ADARSH VIDYA MANDIR,CA ROAD,CA ROAD,NAGPUR,Nagpur,MAHARASHTRA,State,NaN,NaN,B,EAST CENTRE - NAGPUR
4,1004,ADARSH VIDYALAYA,UMRER,UMRER,UMRED,Nagpur,MAHARASHTRA,State,NaN,NaN,B,EAST CENTRE - NAGPUR


In [17]:
#Create a table for all schools using a generic function
create_table_from_df(config['postgres_credentials'], config['db_details']['dbname'], config['db_details']['school_master'], all_schools_ids)

Table successfully created: school_master


***
## Read all students data from various sheets, collate it to create a student master table

1. Student master table will have student name, school name and a unique student id
2. Even if same student appears in different sheets, their information will be captured only once in the student_master table
3. School names to be standardised based on the school_master table
***

## 1. LEAP data

In [18]:
# Load the Leap data
leap_data_df = pd.read_excel('/Users/akshayranade/Documents/Personal/icad_data_management/data/sales/1. 8, 9, 10, 11 Moving All Data.xlsx', '7,8,9,10- Leap 2023-24')

In [19]:
leap_data_df.head()

,Roll No,Student Name,City,Mobile 1,Mobile 2,School,Class
0,121574,Sidhant Bobade,Amravati Centre,NaN,NaN,"Dnyanmata High School Camp road, near IMA hall, Amravati",7th
1,117785,Parth Bonde,Amravati Centre,NaN,NaN,"K.K. Cambridge School, Amravati",7th
2,122267,Ninad Matkar,Amravati Centre,NaN,NaN,"Orchid City International School, Amravati",7th
3,121575,Rachit Ravi Tayade,Amravati Centre,NaN,NaN,"Dnyanmata High School Camp road, near IMA hall, Amravati",7th
4,117780,Pranjal P Kantute,Amravati Centre,NaN,NaN,"K.K. Cambridge School, Amravati",7th


In [20]:
#Clean column names
leap_data_df = clean_column_names(leap_data_df)

In [24]:
leap_data_df['school'] = leap_data_df['school'].apply(lambda x: str(x).replace(',', ' '))

In [25]:
#Clean student names, school names, class
leap_data_df['student_name'] = leap_data_df['student_name'].apply(lambda x: clean_string_data(x))
leap_data_df['city'] = leap_data_df['city'].apply(lambda x: clean_string_data(x))
leap_data_df['school'] = leap_data_df['school'].apply(lambda x: clean_string_data(x))
leap_data_df['class'] = leap_data_df['class'].apply(lambda x: str(x).replace('th', '')).astype(int)

In [26]:
leap_data_df.head()

,roll_no,student_name,city,mobile_1,mobile_2,school,class
0,121574,SIDHANT BOBADE,AMRAVATI CENTRE,NaN,NaN,DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI,7
1,117785,PARTH BONDE,AMRAVATI CENTRE,NaN,NaN,KK CAMBRIDGE SCHOOL AMRAVATI,7
2,122267,NINAD MATKAR,AMRAVATI CENTRE,NaN,NaN,ORCHID CITY INTERNATIONAL SCHOOL AMRAVATI,7
3,121575,RACHIT RAVI TAYADE,AMRAVATI CENTRE,NaN,NaN,DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI,7
4,117780,PRANJAL P KANTUTE,AMRAVATI CENTRE,NaN,NaN,KK CAMBRIDGE SCHOOL AMRAVATI,7


In [27]:
leap_data_df.value_counts('class')

class
10    20317
9      7330
8      4503
7      4034
Name: count, dtype: int64

In [28]:
leap_data_df.shape[0]

36184

In [29]:
#Read the school_master table
school_master = create_df_from_sql(connection_config=config['postgres_credentials'], dbname='icad_db', tbname='school_master')

In [30]:
#Get standard school names & ID's 
leap_data_df['standard_school_name'] = leap_data_df['school'].apply(lambda x: get_standard_school(x, school_master)[0])
leap_data_df['standard_school_addr'] = leap_data_df['school'].apply(lambda x: get_standard_school(x, school_master)[1])
leap_data_df['standard_school_city'] = leap_data_df['school'].apply(lambda x: get_standard_school(x, school_master)[2])
leap_data_df['standard_school_id'] = leap_data_df['school'].apply(lambda x: get_standard_school(x, school_master)[3])

In [31]:
leap_data_df['year'] = '2023-24'

In [32]:
leap_data_df.head()

,roll_no,student_name,city,mobile_1,mobile_2,school,class,standard_school_name,standard_school_addr,standard_school_city,standard_school_id,year
0,121574,SIDHANT BOBADE,AMRAVATI CENTRE,NaN,NaN,DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI,7,DNYANMATA HIGH SCHOOL AMRAVATI,CAMP ROAD 444602,AMRAVATI,1111,2023-24
1,117785,PARTH BONDE,AMRAVATI CENTRE,NaN,NaN,KK CAMBRIDGE SCHOOL AMRAVATI,7,KK CAMBRIDGE SCHOOL,MARDI ROAD AMRAVATI,AMRAVATI,2170,2023-24
2,122267,NINAD MATKAR,AMRAVATI CENTRE,NaN,NaN,ORCHID CITY INTERNATIONAL SCHOOL AMRAVATI,7,ORCHID CITY INTERNATIONAL SCHOOL AMRAVATI,NAVSARI TO AMRAVATI WALGAON ROAD NAVSARI AMRAVATI,AMRAVATI,2239,2023-24
3,121575,RACHIT RAVI TAYADE,AMRAVATI CENTRE,NaN,NaN,DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI,7,DNYANMATA HIGH SCHOOL AMRAVATI,CAMP ROAD 444602,AMRAVATI,1111,2023-24
4,117780,PRANJAL P KANTUTE,AMRAVATI CENTRE,NaN,NaN,KK CAMBRIDGE SCHOOL AMRAVATI,7,KK CAMBRIDGE SCHOOL,MARDI ROAD AMRAVATI,AMRAVATI,2170,2023-24


In [33]:
leap_data_df.shape[0]

36184

In [34]:
#Write the data in table so that it can be referenced in future
create_table_from_df(config['postgres_credentials'], config['db_details']['dbname'], 'stg_leap_data', leap_data_df)

Table successfully created: stg_leap_data


In [ ]:
#Read leap data from database
leap_data_df = create_df_from_sql(connection_config=config['postgres_credentials'], dbname='icad_db', tbname='stg_leap_data')

In [18]:
# leap_data_df.to_csv('/Users/akshayranade/Documents/Personal/icad_data_management/data/sales/cleaned_leap_data.csv')

In [13]:
school_str = 'DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI'

In [20]:
school_master.head()

,school_id,institute_name,address1,address2,city,district,state,board,contact_no,email_address,category,branch_name
0,1000,ABC CONVENT AND HIGH SCHOOL,MANEWADA,MANEWADA,NAGPUR,Nagpur,MAHARASHTRA,State,9096300320,None,B,EAST CENTRE - NAGPUR
1,1001,ADARSH SANSKAR VIDYALAYA HASANBAGH,HASANBAGH,HASANBAGH,NAGPUR,Nagpur,MAHARASHTRA,State,None,None,B,EAST CENTRE - NAGPUR
2,1002,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR,HUDKESHWAR,HUDKESHWAR,NAGPUR,Nagpur,MAHARASHTRA,State,7620264605,None,B,EAST CENTRE - NAGPUR
3,1003,ADARSH VIDYA MANDIR,CA ROAD,CA ROAD,NAGPUR,Nagpur,MAHARASHTRA,State,None,None,B,EAST CENTRE - NAGPUR
4,1004,ADARSH VIDYALAYA,UMRER,UMRER,UMRED,Nagpur,MAHARASHTRA,State,None,None,B,EAST CENTRE - NAGPUR


In [21]:
school_master['school_name_add1'] = school_master['institute_name'] + ' ' + school_master['address1']
school_master['school_name_add2'] = school_master['institute_name'] + ' ' + school_master['address2']
school_master['school_name_add1_city'] = school_master['institute_name'] + ' ' + school_master['address1'] + ' ' + school_master['city']
school_master['school_name_add2_city'] = school_master['institute_name'] + ' ' + school_master['address2'] + ' ' + school_master['city']
school_master['school_name_full_add'] = school_master['institute_name'] + ' ' + school_master['address1'] + ' ' + school_master['address2'] + ' ' + school_master['city'] 

In [22]:
school_master.head()

,school_id,institute_name,address1,address2,city,district,state,board,contact_no,email_address,category,branch_name,school_name_add1,school_name_add2,school_name_add1_city,school_name_add2_city,school_name_full_add
0,1000,ABC CONVENT AND HIGH SCHOOL,MANEWADA,MANEWADA,NAGPUR,Nagpur,MAHARASHTRA,State,9096300320,None,B,EAST CENTRE - NAGPUR,ABC CONVENT AND HIGH SCHOOL MANEWADA,ABC CONVENT AND HIGH SCHOOL MANEWADA,ABC CONVENT AND HIGH SCHOOL MANEWADA NAGPUR,ABC CONVENT AND HIGH SCHOOL MANEWADA NAGPUR,ABC CONVENT AND HIGH SCHOOL MANEWADA MANEWADA NAGPUR
1,1001,ADARSH SANSKAR VIDYALAYA HASANBAGH,HASANBAGH,HASANBAGH,NAGPUR,Nagpur,MAHARASHTRA,State,None,None,B,EAST CENTRE - NAGPUR,ADARSH SANSKAR VIDYALAYA HASANBAGH HASANBAGH,ADARSH SANSKAR VIDYALAYA HASANBAGH HASANBAGH,ADARSH SANSKAR VIDYALAYA HASANBAGH HASANBAGH NAGPUR,ADARSH SANSKAR VIDYALAYA HASANBAGH HASANBAGH NAGPUR,ADARSH SANSKAR VIDYALAYA HASANBAGH HASANBAGH HASANBAGH NAGPUR
2,1002,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR,HUDKESHWAR,HUDKESHWAR,NAGPUR,Nagpur,MAHARASHTRA,State,7620264605,None,B,EAST CENTRE - NAGPUR,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR HUDKESHWAR,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR HUDKESHWAR,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR HUDKESHWAR NAGPUR,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR HUDKESHWAR NAGPUR,ADARSH SANSKAR VIDYALAYA CBSE HUDKESHWAR HUDKESHWAR HUDKESHWAR NAGPUR
3,1003,ADARSH VIDYA MANDIR,CA ROAD,CA ROAD,NAGPUR,Nagpur,MAHARASHTRA,State,None,None,B,EAST CENTRE - NAGPUR,ADARSH VIDYA MANDIR CA ROAD,ADARSH VIDYA MANDIR CA ROAD,ADARSH VIDYA MANDIR CA ROAD NAGPUR,ADARSH VIDYA MANDIR CA ROAD NAGPUR,ADARSH VIDYA MANDIR CA ROAD CA ROAD NAGPUR
4,1004,ADARSH VIDYALAYA,UMRER,UMRER,UMRED,Nagpur,MAHARASHTRA,State,None,None,B,EAST CENTRE - NAGPUR,ADARSH VIDYALAYA UMRER,ADARSH VIDYALAYA UMRER,ADARSH VIDYALAYA UMRER UMRED,ADARSH VIDYALAYA UMRER UMRED,ADARSH VIDYALAYA UMRER UMRER UMRED


In [23]:
school_master[school_master['institute_name'] == 'DNYANMATA HIGH SCHOOL AMRAVATI'].head()

,school_id,institute_name,address1,address2,city,district,state,board,contact_no,email_address,category,branch_name,school_name_add1,school_name_add2,school_name_add1_city,school_name_add2_city,school_name_full_add
111,1111,DNYANMATA HIGH SCHOOL AMRAVATI,CAMP ROAD 444602,AMRAVATI,AMRAVATI,AMRAVATI,MAHARASHTRA,State,7212663263,None,A+,AMRAVATI CENTRE,DNYANMATA HIGH SCHOOL AMRAVATI CAMP ROAD 444602,DNYANMATA HIGH SCHOOL AMRAVATI AMRAVATI,DNYANMATA HIGH SCHOOL AMRAVATI CAMP ROAD 444602 AMRAVATI,DNYANMATA HIGH SCHOOL AMRAVATI AMRAVATI AMRAVATI,DNYANMATA HIGH SCHOOL AMRAVATI CAMP ROAD 444602 AMRAVATI AMRAVATI


In [25]:
school_master[school_master['institute_name'] == 'KK CAMBRIDGE SCHOOL'].head()

,school_id,institute_name,address1,address2,city,district,state,board,contact_no,email_address,category,branch_name,school_name_add1,school_name_add2,school_name_add1_city,school_name_add2_city,school_name_full_add
1165,2162,KK CAMBRIDGE SCHOOL,MARDI ROAD AMRAVATI,AMRAVATI,AMRAVATI,AMRAVATI,MAHARASHTRA,CBSE,None,None,A+,AMRAVATI CENTRE,KK CAMBRIDGE SCHOOL MARDI ROAD AMRAVATI,KK CAMBRIDGE SCHOOL AMRAVATI,KK CAMBRIDGE SCHOOL MARDI ROAD AMRAVATI AMRAVATI,KK CAMBRIDGE SCHOOL AMRAVATI AMRAVATI,KK CAMBRIDGE SCHOOL MARDI ROAD AMRAVATI AMRAVATI AMRAVATI


In [26]:
school_str

'DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI'

In [27]:
#Match a string with a school name
standard_school_name = ''

#Check for exact match with either of the columns
if school_str in school_master['school_name_full_add'].to_list():
    step = 1
    i = school_master['school_name_full_add'].index[school_master['school_name_full_add'].eq(school_str)][0]
    standard_school_name = school_master.loc[i, 'institute_name']
    standard_school_addr = school_master.loc[i, 'school_name_add1_city']
    standard_school_id = school_master.loc[i, 'school_id']
elif school_str in school_master['school_name_add2_city'].to_list():
    step = 2
    i = school_master['school_name_add2_city'].index[school_master['school_name_add2_city'].eq(school_str)][0]
    standard_school_name = school_master.loc[i, 'institute_name']
    standard_school_addr = school_master.loc[i, 'school_name_add1_city']
    standard_school_id = school_master.loc[i, 'school_id']
elif school_str in school_master['school_name_add1_city'].to_list():
    step = 3
    i = school_master['school_name_add1_city'].index[school_master['school_name_add1_city'].eq(school_str)][0]
    standard_school_name = school_master.loc[i, 'institute_name']
    standard_school_addr = school_master.loc[i, 'school_name_add1_city']
    standard_school_id = school_master.loc[i, 'school_id']
elif school_str in school_master['school_name_add2'].to_list():
    step = 4
    i = school_master['school_name_add2'].index[school_master['school_name_add2'].eq(school_str)][0]
    standard_school_name = school_master.loc[i, 'institute_name']
    standard_school_addr = school_master.loc[i, 'school_name_add1_city']
    standard_school_id = school_master.loc[i, 'school_id']
elif school_str in school_master['school_name_add1'].to_list():
    step = 5
    i = school_master['school_name_add1'].index[school_master['school_name_add1'].eq(school_str)][0]
    standard_school_name = school_master.loc[i, 'institute_name']
    standard_school_addr = school_master.loc[i, 'school_name_add1_city']
    standard_school_id = school_master.loc[i, 'school_id']
elif school_str in school_master['institute_name'].to_list():
    step = 6
    i = school_master['institute_name'].index[school_master['institute_name'].eq(school_str)][0]
    standard_school_name = school_master.loc[i, 'institute_name']
    standard_school_addr = school_master.loc[i, 'school_name_add1_city']
    standard_school_id = school_master.loc[i, 'school_id']
else:
    print('N')

N


In [28]:
standard_school_name

''

In [30]:
from difflib import SequenceMatcher
import jellyfish

from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams
from nltk.metrics.distance  import edit_distance

import textdistance

from textdistance import JaroWinkler, Jaro, Cosine, LCSSeq 

In [31]:
master_schools = school_master['school_name_add1_city'].to_list()

match_list_jaro = []
match_list_cosine = []

for s in master_schools:
    match_list_jaro.append(textdistance.jaro_winkler.similarity(school_str, s))
    match_list_cosine.append(textdistance.cosine.similarity(school_str, s))

In [33]:
max(match_list_jaro)

0.9033730158730159

In [38]:
jaro_index = match_list_jaro.index(max(match_list_jaro))

In [40]:
school_master.loc[jaro_index, 'institute_name']

'DNYANMATA HIGH SCHOOL AMRAVATI'

In [34]:
max(match_list_cosine)

0.8728715609439694

In [41]:
cosine_index = match_list_cosine.index(max(match_list_cosine))

In [84]:
school_str = 'DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI'
# standard_str = "DNYANMATA HIGH SCHOOL AMRAVATI CAMP ROAD 444602 AMRAVATI"


# school_str = 'SOMALWAR HIGH SCHOOL RAMDASPETH'
# standard_str = "SOMALWAR HIGH SCHOOL AND JR COLLEGE RAMDASPETH"

In [78]:
#Sequence matcher
s = SequenceMatcher(None, school_str, standard_str)

In [79]:
s.ratio()

0.8051948051948052

In [80]:
#Jellyfish levenstein (replacement based)
jellyfish.levenshtein_distance(school_str, standard_str)

15

In [81]:
#jellyfish jaro (replacement based)
jellyfish.jaro_similarity(school_str, standard_str)

0.8590462833099579

In [82]:
textdistance.jaro_winkler.similarity(school_str, standard_str)

0.9154277699859747

In [83]:
textdistance.cosine.similarity(school_str, standard_str)

0.820922069065183

In [76]:
1-jaccard_distance(set(ngrams(school_str, 3)), set(ngrams(standard_str, 3)))

0.38

In [ ]:
# A function to standardise the school name with reference to the schoool_master table
def standard_school_mapping(school_name_str) -> str:
    #Read school_master data
    school_master = create_df_from_sql(connection_config=config['postgres_credentials'], dbname='icad_db', tbname='school_master')
     

In [107]:
#Split the school name and address
leap_data_df['school'] = leap_data_df['school'].str.strip()

In [112]:
leap_data_df.head()

,roll_no,student_name,city,mobile_1,mobile_2,school,class
0,121574,Sidhant Bobade,Amravati Centre,NaN,NaN,"Dnyanmata High School Camp road, near IMA hall...",7th
1,117785,Parth Bonde,Amravati Centre,NaN,NaN,"K.K. Cambridge School, Amravati",7th
2,122267,Ninad Matkar,Amravati Centre,NaN,NaN,"Orchid City International School, Amravati",7th
3,121575,Rachit Ravi Tayade,Amravati Centre,NaN,NaN,"Dnyanmata High School Camp road, near IMA hall...",7th
4,117780,Pranjal P Kantute,Amravati Centre,NaN,NaN,"K.K. Cambridge School, Amravati",7th


In [122]:
leap_data_df['school_name'] = leap_data_df['school'].apply(lambda x: str(x).split(',')[0] if len(str(x).split(',')) >= 1 else None)

In [124]:
leap_data_df['school_address'] = leap_data_df['school'].apply(lambda x: [' ' + str(x).split(',')[i] for i in (1, len(str(x).split(',')))] if len(str(x).split(',')) > 1 else None)

IndexError: list index out of range

In [115]:
leap_data_df.head()

,roll_no,student_name,city,mobile_1,mobile_2,school,class,school_name
0,121574,Sidhant Bobade,Amravati Centre,NaN,NaN,"Dnyanmata High School Camp road, near IMA hall...",7th,Dnyanmata High School Camp road
1,117785,Parth Bonde,Amravati Centre,NaN,NaN,"K.K. Cambridge School, Amravati",7th,K.K. Cambridge School
2,122267,Ninad Matkar,Amravati Centre,NaN,NaN,"Orchid City International School, Amravati",7th,Orchid City International School
3,121575,Rachit Ravi Tayade,Amravati Centre,NaN,NaN,"Dnyanmata High School Camp road, near IMA hall...",7th,Dnyanmata High School Camp road
4,117780,Pranjal P Kantute,Amravati Centre,NaN,NaN,"K.K. Cambridge School, Amravati",7th,K.K. Cambridge School


In [106]:
PiyushData[PiyushData['school'] == 'None'].head(40)

,slno,student_name,father_name,class,mobileno,school,branch,standard_school_name,standard_school_addr,standard_school_city,standard_school_id


In [25]:
PiyushData = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Piyush Data')

In [26]:
PiyushData.head(5)

,SLNo,Student Name,Father Name,Class,MobileNo,School,Branch
0,1,SARA VIDWANS,MAKARAND VIDWANS,7,NaN,"Bhavan's Bhagwandas Purohit Vidya Mandir, Chinchbhuvan, Nagpur",Head Office - Nagpur
1,2,DEEVENA KHARAT,NITIN KHARAT,7,NaN,"Bhavan's Bhagwandas Purohit Vidya Mandir, Chinchbhuvan, Nagpur",Head Office - Nagpur
2,3,SAKET KATARIA,BHAVESH KATARIA,7,NaN,"Bhavan's Bhagwandas Purohit Vidya Mandir, Chinchbhuvan, Nagpur",Head Office - Nagpur
3,4,SWARNIMA ZADE,DINANATH ZADE,7,NaN,"Bhavan's Bhagwandas Purohit Vidya Mandir, Chinchbhuvan, Nagpur",Head Office - Nagpur
4,5,AARADHYA DHAGE,PRAKASH DHAGE,7,NaN,"Bhavan's Bhagwandas Purohit Vidya Mandir, Chinchbhuvan, Nagpur",Head Office - Nagpur


In [27]:
PiyushData = clean_column_names(PiyushData)

In [28]:
PiyushData['school'] = PiyushData['school'].apply(lambda x: str(x).replace(',', ' '))

In [29]:
PiyushData.head(5)

,slno,student_name,father_name,class,mobileno,school,branch
0,1,SARA VIDWANS,MAKARAND VIDWANS,7,NaN,Bhavan's Bhagwandas Purohit Vidya Mandir Chinchbhuvan Nagpur,Head Office - Nagpur
1,2,DEEVENA KHARAT,NITIN KHARAT,7,NaN,Bhavan's Bhagwandas Purohit Vidya Mandir Chinchbhuvan Nagpur,Head Office - Nagpur
2,3,SAKET KATARIA,BHAVESH KATARIA,7,NaN,Bhavan's Bhagwandas Purohit Vidya Mandir Chinchbhuvan Nagpur,Head Office - Nagpur
3,4,SWARNIMA ZADE,DINANATH ZADE,7,NaN,Bhavan's Bhagwandas Purohit Vidya Mandir Chinchbhuvan Nagpur,Head Office - Nagpur
4,5,AARADHYA DHAGE,PRAKASH DHAGE,7,NaN,Bhavan's Bhagwandas Purohit Vidya Mandir Chinchbhuvan Nagpur,Head Office - Nagpur


In [30]:
#Clean student names, school names, class
PiyushData['student_name'] = PiyushData['student_name'].apply(lambda x: clean_string_data(x))
PiyushData['school'] = PiyushData['school'].apply(lambda x: clean_string_data(x))
PiyushData['class'] = PiyushData['class'].apply(lambda x: str(x).replace('th', '')).astype(int)

In [31]:
PiyushData.value_counts('class')

class
7     4277
8     4161
9     4033
10    3650
Name: count, dtype: int64

In [32]:
school_master = all_schools_ids

In [33]:
#Get standard school names & ID's 
PiyushData['standard_school_name'] = PiyushData['school'].apply(lambda x: get_standard_school(x, school_master)[0])
PiyushData['standard_school_addr'] = PiyushData['school'].apply(lambda x: get_standard_school(x, school_master)[1])
PiyushData['standard_school_city'] = PiyushData['school'].apply(lambda x: get_standard_school(x, school_master)[2])
PiyushData['standard_school_id'] = PiyushData['school'].apply(lambda x: get_standard_school(x, school_master)[3])

In [35]:
PiyushData.head(5)

,slno,student_name,father_name,class,mobileno,school,branch,standard_school_name,standard_school_addr,standard_school_city,standard_school_id
0,1,SARA VIDWANS,MAKARAND VIDWANS,7,NaN,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR CHINCHBHUVAN NAGPUR,Head Office - Nagpur,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,SHRIKRISHNA NAGAR,NAGPUR,3530
1,2,DEEVENA KHARAT,NITIN KHARAT,7,NaN,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR CHINCHBHUVAN NAGPUR,Head Office - Nagpur,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,SHRIKRISHNA NAGAR,NAGPUR,3530
2,3,SAKET KATARIA,BHAVESH KATARIA,7,NaN,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR CHINCHBHUVAN NAGPUR,Head Office - Nagpur,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,SHRIKRISHNA NAGAR,NAGPUR,3530
3,4,SWARNIMA ZADE,DINANATH ZADE,7,NaN,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR CHINCHBHUVAN NAGPUR,Head Office - Nagpur,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,SHRIKRISHNA NAGAR,NAGPUR,3530
4,5,AARADHYA DHAGE,PRAKASH DHAGE,7,NaN,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR CHINCHBHUVAN NAGPUR,Head Office - Nagpur,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,SHRIKRISHNA NAGAR,NAGPUR,3530


In [44]:
import pandas as pd

excel_path = '/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx'  # Replace with your file path

xls = pd.ExcelFile(excel_path)
sheet_names = xls.sheet_names

print("Sheet names:", sheet_names)

Sheet names: ['7,8,9,10- Leap 2023-24', 'Piyush Data', 'Olympiad-2019 Ph-2', 'Olympiad-2020  Ph-2', 'Olympiad-2021', 'Olympiad-Feb 23', 'Olympiad-June 23', 'Talent Data', 'CDS Data', ' VNIT Axis-23 Mathamaze', ' VNIT Axis-23 Mathemaze round 1', ' VNIT Axis-23 Mathemaze round 2', ' VNIT Axis-23 except Mathamaze', "JUNIOR SCIENTIST '23 "]


In [193]:
def replace_generic_with_list_items(code_template, replacements):
    """
    Replaces <Generic> in the code_template with each value from replacements list, one per version.

    :param code_template: String, multiline Python code with <Generic> placeholders
    :param replacements: List of strings to replace <Generic> with
    :return: List of modified code strings
    """
    result = []
    for item in replacements:
        modified_code = code_template.replace('<Generic>', item)
        result.append(modified_code)
    return result

# Example usage
code_block = """
<Generic>.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/<Generic>.csv', encoding='utf-8-sig')
"""

dataframes = ['leapData','Olympiad2019Ph2','Olympiad2020Ph2','Olympiad2021','OlympiadFeb23','OlympiadJune23','TalentData','CDSData','VNITAxis23Mathamaze','VNITAxis23Mathemazeround1','VNITAxis23Mathemazeround2','VNITAxis23exceptMathamaze']

modified_blocks = replace_generic_with_list_items(code_block, dataframes)

# To print all versions
for idx, block in enumerate(modified_blocks, start=1):
    print(block)


leapData.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/leapData.csv', encoding='utf-8-sig')


Olympiad2019Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/Olympiad2019Ph2.csv', encoding='utf-8-sig')


Olympiad2020Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/Olympiad2020Ph2.csv', encoding='utf-8-sig')


Olympiad2021.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/Olympiad2021.csv', encoding='utf-8-sig')


OlympiadFeb23.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/OlympiadFeb23.csv', encoding='utf-8-sig')


OlympiadJune23.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/OlympiadJune23.csv', encoding='utf-8-sig')


TalentData.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/TalentData.csv', encoding='utf-8-sig')


CDSData.to_csv('/Users/prathampandey/Documents/C

In [43]:

Olympiad2019Ph2 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Olympiad-2019 Ph-2')
Olympiad2020Ph2 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Olympiad-2020  Ph-2')
Olympiad2021 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Olympiad-2021')
OlympiadFeb23 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Olympiad-Feb 23')
OlympiadJune23 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Olympiad-June 23')
TalentData = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Talent Data')
CDSData = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'CDS Data')
VNITAxis23Mathamaze = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', ' VNIT Axis-23 Mathamaze')
VNITAxis23Mathemazeround1 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', ' VNIT Axis-23 Mathemaze round 1')
VNITAxis23Mathemazeround2 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', ' VNIT Axis-23 Mathemaze round 2')
VNITAxis23exceptMathamaze = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', ' VNIT Axis-23 except Mathamaze')

In [46]:
Olympiad2019Ph2 = clean_column_names(Olympiad2019Ph2)


Olympiad2020Ph2 = clean_column_names(Olympiad2020Ph2)


Olympiad2021 = clean_column_names(Olympiad2021)


OlympiadFeb23 = clean_column_names(OlympiadFeb23)


OlympiadJune23 = clean_column_names(OlympiadJune23)


TalentData = clean_column_names(TalentData)


CDSData = clean_column_names(CDSData)


VNITAxis23Mathamaze = clean_column_names(VNITAxis23Mathamaze)


VNITAxis23Mathemazeround1 = clean_column_names(VNITAxis23Mathemazeround1)


VNITAxis23Mathemazeround2 = clean_column_names(VNITAxis23Mathemazeround2)


VNITAxis23exceptMathamaze = clean_column_names(VNITAxis23exceptMathamaze)

In [50]:
Olympiad2019Ph2.rename(columns={'name_of_student': 'student_name','school_name': 'school'}, inplace=True)
Olympiad2019Ph2.head(5)

,sr_no,student_name,class,school,parents_ph_no
0,1,Vidit Ajunash Kandey,9,Mount Litera be School,NaN
1,2,Rishika Dobabrata Mukherjee,9,Bhavan's NTPC Vidya Mandir,NaN
2,3,Aditya Anil Verma,9,Sacred Heart Academy,NaN
3,4,Siddharth Abhivilas Nakhate,9,Narayana Vidyalaya,NaN
4,5,Pranjali Namdev Rathod,9,Bhavan's NTPC Vidya Mandir,NaN


In [51]:
Olympiad2020Ph2.rename(columns={'name_of_student': 'student_name','school_name': 'school'}, inplace=True)
Olympiad2020Ph2.head(5)

,sr_no,student_name,class,school,students_ph_no
0,1,Khushad Katiyar,8,"Delhi Public School, Nagpur",NaN
1,2,Ishan Patil,8,"Delhi Public School, Nagpur",NaN
2,3,Advait Lade,8,"Bhavan's B.P. Vidya Mandir, Nagpur",NaN
3,4,Chirag Goenka,8,"The Achievers School, Nagpur",NaN
4,5,Ishwari Bhande,8,"Rahi Public School, Nagpur",NaN


In [54]:
Olympiad2021.rename(columns={'name_of_student': 'student_name','school_name': 'school'}, inplace=True)
Olympiad2021.head(5)

,sr_no,student_name,class,school,students_phone_no,parents_phone_no,email_id,nco__nso__imo,address
0,1,Ambar Arpita Gupta,7,"Delhi Public School, Nagpur",NaN,NaN,NaN,NSO,149-A Tirumal Wardhman nagar Nagpur
1,2,Saanvi Hitesh Khetan,7,"Delhi Public School, Nagpur",NaN,NaN,NaN,NSO,"Flat no. 201, Shri krishna Apartment 30 AVG Layout Lakadganj, Nagpur"
2,3,Vayun Agrawal,7,"Delhi Public School, Nagpur",NaN,NaN,NaN,NSO,NaN
3,4,Devanshi P. Lalwani,7,"Priyadarshini Nagpur Public School, Nagpur",NaN,NaN,NaN,NSO,"465 Keshranand Mahal, Nagpur 440032"
4,5,Jay Suchak,7,"Priyadarshini Nagpur Public School, Nagpur",NaN,NaN,NaN,NSO,"389, Ashirwad Bawan, Nagpur"


In [56]:
OlympiadFeb23.rename(columns={'name_of_student': 'student_name','school_name': 'school'}, inplace=True)
OlympiadFeb23.head(5)

,sr_no,student_name,class,section,student,parent,year,address,school,centre,year
0,1,Daanish Vaidya,3,A,NaN,NaN,2022-23,"Katol Road, Nagpur",-,NaN,2023-02-01
1,2,Akshad R. Lanjewar,3,NaN,NaN,NaN,2022-23,"Vinoba Bhave Nagar, Nagpur","Aadarsh Sanskar Vidyalaya, Nagpur",East Centre- Nagpur,2023-02-01
2,3,Madhura M. Sawarkar,3,NaN,NaN,NaN,IMO-2022-23,"Sahakar Nagar, Kharabi Road, Nagpur.","Aadarsh Sanskar Vidyalaya, Nagpur",East Centre- Nagpur,2023-02-01
3,4,Yukti M. Duragkar,3,NaN,NaN,NaN,IMO-2022-23,"Plot No. 58, Lakkalyan Nagar, Wathoda, Nagpur.","Aadarsh Sanskar Vidyalaya, Nagpur",East Centre- Nagpur,2023-02-01
4,5,Harshali Kawale,3,NaN,977640251,NaN,IMO-2022-23,"Sahakar Nagar, Kharabi Road, Nagpur.","Aadarsh Sanskar Vidyalaya, Nagpur",East Centre- Nagpur,2023-02-01


In [58]:
OlympiadJune23.rename(columns={'name_of_student': 'student_name','school_name': 'school'}, inplace=True)   
OlympiadJune23.head(5)


,sr_no,student_name,class,section,mobile_no,igko,ieo,nso,imo,nco,isso,iso,school,centre,year
0,1,Archana Anil Verma,7,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,Tip Top Convent Swawalambi Nagar,NaN,2023
1,2,Adity Shivshankar Kanoje,7,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,Tip Top Convent Swawalambi Nagar,NaN,2023
2,3,Devshree Vishes Tajne,7,NaN,NaN,Yes,NaN,NaN,Yes,NaN,NaN,NaN,Tip Top Convent Swawalambi Nagar,NaN,2023
3,4,Vidisha Vaibhav Koyande,7,NaN,NaN,NaN,Yes,Yes,Yes,NaN,NaN,NaN,Tip Top Convent Swawalambi Nagar,NaN,2023
4,5,Sachi Satish Bhagat,7,NaN,NaN,Yes,NaN,NaN,NaN,NaN,NaN,NaN,Tip Top Convent Swawalambi Nagar,NaN,2023


In [67]:

TalentData.rename(columns={'name_of_student': 'student_name','school_name': 'school','standard':'class'}, inplace=True)  
TalentData.head(5)

,sr_no,city,student_name,school,branch,class,mobile_no1,mobile_no2,exam,address
0,1,AMRAVATI,Tasmay Manoj Shingane,"Podar International School, Amravati.",Amravati Centre,8,NaN,NaN,GANIT PRADNYA EXAMINATION 2023,NaN
1,2,AMRAVATI,Aryan Ashish Harkut,"TomoaeEnglish Primary English School, Amt.",Amravati Centre,8,NaN,NaN,GANIT PRADNYA EXAMINATION 2023,NaN
2,3,AMRAVATI,Krushna Santosh Giri,Avent Grade Public School.Paratwada.,Amravati Centre,8,NaN,NaN,GANIT PRADNYA EXAMINATION 2023,NaN
3,4,AMRAVATI,Kartik Ashish Bang,"Narayana Vidyalayam, Amravati.",Amravati Centre,8,NaN,NaN,GANIT PRADNYA EXAMINATION 2023,NaN
4,5,AMRAVATI,Abhiram Anand Dhote,"Podar International School, Amravati.",Amravati Centre,8,NaN,NaN,GANIT PRADNYA EXAMINATION 2023,NaN


In [ ]:
CDSData.head(5)
#Need to check

,sno,name_of_students,mobile_no,class,centre
0,486,Aldak Dharini,NaN,5,Sadar Centre - Nagpur
1,487,Ansari Ishrat,NaN,5,Sadar Centre - Nagpur
2,488,Borkar Nisha,NaN,5,Sadar Centre - Nagpur
3,489,Bouddh Naitikta,NaN,5,Sadar Centre - Nagpur
4,490,Damble Nidhi,NaN,5,Sadar Centre - Nagpur


In [81]:
VNITAxis23Mathamaze.rename(columns={'name': 'student_name'}, inplace=True)
VNITAxis23Mathamaze.head(5)

,sr_no,student_name,class,phone,school
0,1,Jivti khurana,5.0,NaN,Bhavans B.P Vidya Mandir
1,2,SWARA AMIT MUDALIAR,5.0,NaN,"BHAWAN'S BHAGWANDAS PUROHIT VIDYA MANDIR, TRIMURITY NAGAR,NAGPUR"
2,3,Hazel Pahuja,5.0,NaN,Centre Point School
3,4,Advika Lokhande,5.0,NaN,Delhi Public school
4,5,JIYANSH SUKHADIA,5.0,NaN,DPS MIHAN NAGPUR


In [82]:
VNITAxis23Mathemazeround1.rename(columns={'name': 'student_name'}, inplace=True)
VNITAxis23Mathemazeround1.head(5)

,sr_no,student_name,class,phone,school,marks
0,1,Aarnav Mange,5.0,NaN,Bhavans BP Vidya Mandir,2
1,2,akshaj rathod,5.0,NaN,bvm trimurti,1
2,3,anika jaiswal,5.0,NaN,bhavans chinchavan,2
3,4,Anuja Mahesh Varma,5.0,NaN,Bhavans Bhagvandas Purohit Vidya Mandi Trimurti nagar Nagpur,3
4,5,Arnav Mawle,5.0,NaN,Narayana Vidyalaya chinch bhawan,1


In [115]:
VNITAxis23Mathemazeround2.rename(columns={'name': 'student_name'}, inplace=True)    
VNITAxis23Mathemazeround2.head(5)

,sr_no,student_name,class,phone,school,marks,standard_school_name,standard_school_addr,standard_school_city,standard_school_id
0,1,AARNAV MANGE,5,NaN,BHAVANS BP VIDYA MANDIR,-7.0,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,CHINCHBHAVAN NAGPUR,NAGPUR,2633
1,2,AKSHAJ RATHOD,5,NaN,BVM TRIMURTI,6.0,SPM CONVENT,ARNI,ARNI,2285
2,3,ANIKA JAISWAL,5,NaN,BHAVANS CHINCHAVAN,1.0,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,CHINCHBHAVAN NAGPUR,NAGPUR,2633
3,4,ANUJA MAHESH VARMA,5,NaN,BHAVANS BHAGVANDAS PUROHIT VIDYA MANDI TRIMURTI NAGAR NAGPUR,1.0,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR TRIMUTI NAGAR,TRIMURTI NAGAR,NAGPUR,1388
4,5,ARNAV MAWLE,5,NaN,NARAYANA VIDYALAYA CHINCH BHAWAN,-4.0,NARAYANA VIDYALAYAM,WARDHA ROAD NAGPUR,NAGPUR,2629


In [84]:
VNITAxis23exceptMathamaze.rename(columns={'name': 'student_name'}, inplace=True)
VNITAxis23exceptMathamaze.head(5)

,sr_no,student_name,class,phone,school
0,1,Aabhas Sur,5.0,NaN,St. Francis De Sales High School
1,2,Aadya Sourav Mulmele,5.0,NaN,Somalwar Nikalas High School
2,3,Adheesh Mithun Deshmukh,5.0,NaN,Tip top Convent
3,4,adhesh deshmukh,5.0,NaN,tip top
4,5,Aditya Gupta,5.0,NaN,Tip Top Convent


In [102]:
#Clean student names, school names, class
Olympiad2019Ph2['student_name'] = Olympiad2019Ph2['student_name'].apply(lambda x: clean_string_data(x))
Olympiad2019Ph2['school'] = Olympiad2019Ph2['school'].apply(lambda x: clean_string_data(x))
Olympiad2019Ph2['class'] = Olympiad2019Ph2['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
Olympiad2020Ph2['student_name'] = Olympiad2020Ph2['student_name'].apply(lambda x: clean_string_data(x))
Olympiad2020Ph2['school'] = Olympiad2020Ph2['school'].apply(lambda x: clean_string_data(x))
Olympiad2020Ph2['class'] = Olympiad2020Ph2['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
Olympiad2021['student_name'] = Olympiad2021['student_name'].apply(lambda x: clean_string_data(x))
Olympiad2021['school'] = Olympiad2021['school'].apply(lambda x: clean_string_data(x))
Olympiad2021['class'] = Olympiad2021['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
OlympiadFeb23['student_name'] = OlympiadFeb23['student_name'].apply(lambda x: clean_string_data(x))
OlympiadFeb23['school'] = OlympiadFeb23['school'].apply(lambda x: clean_string_data(x))
OlympiadFeb23['class'] = OlympiadFeb23['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
OlympiadJune23['student_name'] = OlympiadJune23['student_name'].apply(lambda x: clean_string_data(x))
OlympiadJune23['school'] = OlympiadJune23['school'].apply(lambda x: clean_string_data(x))
OlympiadJune23['class'] = OlympiadJune23['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
TalentData['student_name'] = TalentData['student_name'].apply(lambda x: clean_string_data(x))
TalentData['city'] = TalentData['city'].apply(lambda x: clean_string_data(x))
TalentData['school'] = TalentData['school'].apply(lambda x: clean_string_data(x))
TalentData['class'] = TalentData['class'].apply(lambda x: str(x).replace('th', '')).astype(int)



#Clean student names, school names, class
VNITAxis23Mathamaze['student_name'] = VNITAxis23Mathamaze['student_name'].apply(lambda x: clean_string_data(x))
VNITAxis23Mathamaze['school'] = VNITAxis23Mathamaze['school'].apply(lambda x: clean_string_data(x))
VNITAxis23Mathamaze['class'] = VNITAxis23Mathamaze['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
VNITAxis23Mathemazeround1['student_name'] = VNITAxis23Mathemazeround1['student_name'].apply(lambda x: clean_string_data(x))
VNITAxis23Mathemazeround1['school'] = VNITAxis23Mathemazeround1['school'].apply(lambda x: clean_string_data(x))
VNITAxis23Mathemazeround1['class'] = VNITAxis23Mathemazeround1['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
VNITAxis23Mathemazeround2['student_name'] = VNITAxis23Mathemazeround2['student_name'].apply(lambda x: clean_string_data(x))
VNITAxis23Mathemazeround2['school'] = VNITAxis23Mathemazeround2['school'].apply(lambda x: clean_string_data(x))
VNITAxis23Mathemazeround2['class'] = VNITAxis23Mathemazeround2['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


#Clean student names, school names, class
VNITAxis23exceptMathamaze['student_name'] = VNITAxis23exceptMathamaze['student_name'].apply(lambda x: clean_string_data(x))
VNITAxis23exceptMathamaze['school'] = VNITAxis23exceptMathamaze['school'].apply(lambda x: clean_string_data(x))
VNITAxis23exceptMathamaze['class'] = VNITAxis23exceptMathamaze['class'].apply(lambda x: str(x).replace('th', '')).astype(int)


In [79]:
OlympiadFeb23['class'].unique()


array([ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12,  0])

In [99]:
VNITAxis23Mathamaze['class'].unique()

array([ 5,  6,  7,  8,  9, 10, 11, 12])

In [ ]:
OlympiadFeb23['class'] = OlympiadFeb23['class'] \
    .replace('-', np.nan) \
    .dropna() \
    .apply(lambda x: str(x).replace('th', '')) \
    .astype(float).astype(int)



In [101]:
VNITAxis23Mathamaze = VNITAxis23Mathamaze.dropna(subset=['class'])
VNITAxis23Mathamaze['class'] = VNITAxis23Mathamaze['class'].astype(int)

VNITAxis23Mathemazeround1 = VNITAxis23Mathemazeround1.dropna(subset=['class'])
VNITAxis23Mathemazeround1['class'] = VNITAxis23Mathemazeround1['class'].astype(int)
VNITAxis23Mathemazeround2 = VNITAxis23Mathemazeround2.dropna(subset=['class'])
VNITAxis23Mathemazeround2['class'] = VNITAxis23Mathemazeround2['class'].astype(int)
VNITAxis23exceptMathamaze = VNITAxis23exceptMathamaze.dropna(subset=['class'])
VNITAxis23exceptMathamaze['class'] = VNITAxis23exceptMathamaze['class'].astype(int)

In [104]:
Olympiad2019Ph2['standard_school_name'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[0])
Olympiad2019Ph2['standard_school_addr'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[1])
Olympiad2019Ph2['standard_school_city'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[2])
Olympiad2019Ph2['standard_school_id'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[3])


Olympiad2020Ph2['standard_school_name'] = Olympiad2020Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[0])
Olympiad2020Ph2['standard_school_addr'] = Olympiad2020Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[1])
Olympiad2020Ph2['standard_school_city'] = Olympiad2020Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[2])
Olympiad2020Ph2['standard_school_id'] = Olympiad2020Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[3])


Olympiad2021['standard_school_name'] = Olympiad2021['school'].apply(lambda x: get_standard_school(x, school_master)[0])
Olympiad2021['standard_school_addr'] = Olympiad2021['school'].apply(lambda x: get_standard_school(x, school_master)[1])
Olympiad2021['standard_school_city'] = Olympiad2021['school'].apply(lambda x: get_standard_school(x, school_master)[2])
Olympiad2021['standard_school_id'] = Olympiad2021['school'].apply(lambda x: get_standard_school(x, school_master)[3])


OlympiadFeb23['standard_school_name'] = OlympiadFeb23['school'].apply(lambda x: get_standard_school(x, school_master)[0])
OlympiadFeb23['standard_school_addr'] = OlympiadFeb23['school'].apply(lambda x: get_standard_school(x, school_master)[1])
OlympiadFeb23['standard_school_city'] = OlympiadFeb23['school'].apply(lambda x: get_standard_school(x, school_master)[2])
OlympiadFeb23['standard_school_id'] = OlympiadFeb23['school'].apply(lambda x: get_standard_school(x, school_master)[3])


OlympiadJune23['standard_school_name'] = OlympiadJune23['school'].apply(lambda x: get_standard_school(x, school_master)[0])
OlympiadJune23['standard_school_addr'] = OlympiadJune23['school'].apply(lambda x: get_standard_school(x, school_master)[1])
OlympiadJune23['standard_school_city'] = OlympiadJune23['school'].apply(lambda x: get_standard_school(x, school_master)[2])
OlympiadJune23['standard_school_id'] = OlympiadJune23['school'].apply(lambda x: get_standard_school(x, school_master)[3])


TalentData['standard_school_name'] = TalentData['school'].apply(lambda x: get_standard_school(x, school_master)[0])
TalentData['standard_school_addr'] = TalentData['school'].apply(lambda x: get_standard_school(x, school_master)[1])
TalentData['standard_school_city'] = TalentData['school'].apply(lambda x: get_standard_school(x, school_master)[2])
TalentData['standard_school_id'] = TalentData['school'].apply(lambda x: get_standard_school(x, school_master)[3])


# CDSData['standard_school_name'] = CDSData['school'].apply(lambda x: get_standard_school(x, school_master)[0])
# CDSData['standard_school_addr'] = CDSData['school'].apply(lambda x: get_standard_school(x, school_master)[1])
# CDSData['standard_school_city'] = CDSData['school'].apply(lambda x: get_standard_school(x, school_master)[2])
# CDSData['standard_school_id'] = CDSData['school'].apply(lambda x: get_standard_school(x, school_master)[3])


VNITAxis23Mathamaze['standard_school_name'] = VNITAxis23Mathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[0])
VNITAxis23Mathamaze['standard_school_addr'] = VNITAxis23Mathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[1])
VNITAxis23Mathamaze['standard_school_city'] = VNITAxis23Mathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[2])
VNITAxis23Mathamaze['standard_school_id'] = VNITAxis23Mathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[3])


VNITAxis23Mathemazeround1['standard_school_name'] = VNITAxis23Mathemazeround1['school'].apply(lambda x: get_standard_school(x, school_master)[0])
VNITAxis23Mathemazeround1['standard_school_addr'] = VNITAxis23Mathemazeround1['school'].apply(lambda x: get_standard_school(x, school_master)[1])
VNITAxis23Mathemazeround1['standard_school_city'] = VNITAxis23Mathemazeround1['school'].apply(lambda x: get_standard_school(x, school_master)[2])
VNITAxis23Mathemazeround1['standard_school_id'] = VNITAxis23Mathemazeround1['school'].apply(lambda x: get_standard_school(x, school_master)[3])


VNITAxis23Mathemazeround2['standard_school_name'] = VNITAxis23Mathemazeround2['school'].apply(lambda x: get_standard_school(x, school_master)[0])
VNITAxis23Mathemazeround2['standard_school_addr'] = VNITAxis23Mathemazeround2['school'].apply(lambda x: get_standard_school(x, school_master)[1])
VNITAxis23Mathemazeround2['standard_school_city'] = VNITAxis23Mathemazeround2['school'].apply(lambda x: get_standard_school(x, school_master)[2])
VNITAxis23Mathemazeround2['standard_school_id'] = VNITAxis23Mathemazeround2['school'].apply(lambda x: get_standard_school(x, school_master)[3])


VNITAxis23exceptMathamaze['standard_school_name'] = VNITAxis23exceptMathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[0])
VNITAxis23exceptMathamaze['standard_school_addr'] = VNITAxis23exceptMathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[1])
VNITAxis23exceptMathamaze['standard_school_city'] = VNITAxis23exceptMathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[2])
VNITAxis23exceptMathamaze['standard_school_id'] = VNITAxis23exceptMathamaze['school'].apply(lambda x: get_standard_school(x, school_master)[3])


In [111]:
Olympiad2019Ph2 = Olympiad2019Ph2[Olympiad2019Ph2['school'].notna() & (Olympiad2019Ph2['school'].str.strip() != '')]


Olympiad2020Ph2 = Olympiad2020Ph2[Olympiad2020Ph2['school'].notna() & (Olympiad2020Ph2['school'].str.strip() != '')]


Olympiad2021 = Olympiad2021[Olympiad2021['school'].notna() & (Olympiad2021['school'].str.strip() != '')]


OlympiadFeb23 = OlympiadFeb23[OlympiadFeb23['school'].notna() & (OlympiadFeb23['school'].str.strip() != '')]


OlympiadJune23 = OlympiadJune23[OlympiadJune23['school'].notna() & (OlympiadJune23['school'].str.strip() != '')]


TalentData = TalentData[TalentData['school'].notna() & (TalentData['school'].str.strip() != '')]


VNITAxis23Mathamaze = VNITAxis23Mathamaze[VNITAxis23Mathamaze['school'].notna() & (VNITAxis23Mathamaze['school'].str.strip() != '')]


VNITAxis23Mathemazeround1 = VNITAxis23Mathemazeround1[VNITAxis23Mathemazeround1['school'].notna() & (VNITAxis23Mathemazeround1['school'].str.strip() != '')]


VNITAxis23Mathemazeround2 = VNITAxis23Mathemazeround2[VNITAxis23Mathemazeround2['school'].notna() & (VNITAxis23Mathemazeround2['school'].str.strip() != '')]


VNITAxis23exceptMathamaze = VNITAxis23exceptMathamaze[VNITAxis23exceptMathamaze['school'].notna() & (VNITAxis23exceptMathamaze['school'].str.strip() != '')]


In [114]:
leapData = pd.read_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/cleaned_leap_data.csv')

In [ ]:
Olympiad2019Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/Olympiad2019Ph2_cleaned.csv', encoding='utf-8-sig')

Olympiad2020Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/Olympiad2020Ph2_cleaned.csv', encoding='utf-8-sig')


Olympiad2021.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/Olympiad2021_cleaned.csv', encoding='utf-8-sig')


OlympiadFeb23.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/OlympiadFeb23_cleaned.csv', encoding='utf-8-sig')


OlympiadJune23.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/OlympiadJune23_cleaned.csv', encoding='utf-8-sig')


TalentData.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/TalentData_cleaned.csv', encoding='utf-8-sig')


CDSData.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/CDSData_cleaned.csv', encoding='utf-8-sig')


VNITAxis23Mathamaze.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/VNITAxis23Mathamaze_cleaned.csv', encoding='utf-8-sig')


VNITAxis23Mathemazeround1.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/VNITAxis23Mathemazeround1_cleaned.csv', encoding='utf-8-sig')


VNITAxis23Mathemazeround2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/VNITAxis23Mathemazeround2_cleaned.csv', encoding='utf-8-sig')


VNITAxis23exceptMathamaze.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/VNITAxis23exceptMathamaze_cleaned.csv', encoding='utf-8-sig')

In [116]:
import pandas as pd

excel_path = '/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx'  # Replace with your file path

xls = pd.ExcelFile(excel_path)
sheet_names = xls.sheet_names

print("Sheet names:", sheet_names)

Sheet names: ['7,8,9,10- Leap 2023-24', 'Piyush Data', 'Olympiad-2019 Ph-2', 'Olympiad-2020  Ph-2', 'Olympiad-2021', 'Olympiad-Feb 23', 'Olympiad-June 23', 'Talent Data', 'CDS Data', ' VNIT Axis-23 Mathamaze', ' VNIT Axis-23 Mathemaze round 1', ' VNIT Axis-23 Mathemaze round 2', ' VNIT Axis-23 except Mathamaze', "JUNIOR SCIENTIST '23 "]


In [117]:
leapData.head(5)

,Unnamed: 0,roll_no,student_name,city,mobile_1,mobile_2,school,class,standard_school_name,standard_school_id
0,0,121574,SIDHANT BOBADE,AMRAVATI CENTRE,NaN,NaN,DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI,7,DNYANMATA HIGH SCHOOL AMRAVATI,1111
1,1,117785,PARTH BONDE,AMRAVATI CENTRE,NaN,NaN,KK CAMBRIDGE SCHOOL AMRAVATI,7,KK CAMBRIDGE SCHOOL,2162
2,2,122267,NINAD MATKAR,AMRAVATI CENTRE,NaN,NaN,ORCHID CITY INTERNATIONAL SCHOOL AMRAVATI,7,ORCHID CITY INTERNATIONAL SCHOOL AMRAVATI,2232
3,3,121575,RACHIT RAVI TAYADE,AMRAVATI CENTRE,NaN,NaN,DNYANMATA HIGH SCHOOL CAMP ROAD NEAR IMA HALL AMRAVATI,7,DNYANMATA HIGH SCHOOL AMRAVATI,1111
4,4,117780,PRANJAL P KANTUTE,AMRAVATI CENTRE,NaN,NaN,KK CAMBRIDGE SCHOOL AMRAVATI,7,KK CAMBRIDGE SCHOOL,2162


In [118]:
final_columns = ['student_name', 'standard_school_name', 'standard_school_id', 'class']

In [121]:
leapData= leapData[final_columns]
leapData['exam']='LEAP'
leapData.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,SIDHANT BOBADE,DNYANMATA HIGH SCHOOL AMRAVATI,1111,7,LEAP
1,PARTH BONDE,KK CAMBRIDGE SCHOOL,2162,7,LEAP
2,NINAD MATKAR,ORCHID CITY INTERNATIONAL SCHOOL AMRAVATI,2232,7,LEAP
3,RACHIT RAVI TAYADE,DNYANMATA HIGH SCHOOL AMRAVATI,1111,7,LEAP
4,PRANJAL P KANTUTE,KK CAMBRIDGE SCHOOL,2162,7,LEAP


In [123]:
PiyushData = PiyushData[final_columns]
PiyushData['exam']='Piyush'
PiyushData.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,SARA VIDWANS,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,3530,7,Piyush
1,DEEVENA KHARAT,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,3530,7,Piyush
2,SAKET KATARIA,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,3530,7,Piyush
3,SWARNIMA ZADE,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,3530,7,Piyush
4,AARADHYA DHAGE,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR,3530,7,Piyush


In [151]:

Olympiad2019Ph2['class'].unique()

array([14., 15., nan])

In [153]:
Olympiad2020Ph2['class'] = Olympiad2020Ph2['class'].astype(float) + 4
Olympiad2020Ph2.head(5)

,sr_no,student_name,class,school,students_ph_no,standard_school_name,standard_school_addr,standard_school_city,standard_school_id
0,1,KHUSHAD KATIYAR,13.0,DELHI PUBLIC SCHOOL NAGPUR,NaN,DELHI PUBLIC SCHOOL,KAMPTEE ROAD OPP11KMS MILESTONE AT POST KHAIRI,NAGPUR,1332
1,2,ISHAN PATIL,13.0,DELHI PUBLIC SCHOOL NAGPUR,NaN,DELHI PUBLIC SCHOOL,KAMPTEE ROAD OPP11KMS MILESTONE AT POST KHAIRI,NAGPUR,1332
2,3,ADVAIT LADE,13.0,BHAVANS BP VIDYA MANDIR NAGPUR,NaN,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,CHINCHBHAVAN NAGPUR,NAGPUR,2633
3,4,CHIRAG GOENKA,13.0,THE ACHIEVERS SCHOOL NAGPUR,NaN,THE ACHIEVERS SCHOOL,WARODA VILLAGE OUTER RING ROAD NAGPUR MAHARASHTRA 441108,NAGPUR,2270
4,5,ISHWARI BHANDE,13.0,RAHI PUBLIC SCHOOL NAGPUR,NaN,RAHI PUBLIC SCHOOL,ZADE LE OUT JAITALA ROAD JAITALA NAGPUR 440036,NAGPUR,2661


In [ ]:
Olympiad2020Ph2 = Olympiad2020Ph2[final_columns]
Olympiad2020Ph2['exam']='Olympiad2020'
Olympiad2020Ph2.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,KHUSHAD KATIYAR,DELHI PUBLIC SCHOOL,1332,8,Olympiad2020
1,ISHAN PATIL,DELHI PUBLIC SCHOOL,1332,8,Olympiad2020
2,ADVAIT LADE,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,2633,8,Olympiad2020
3,CHIRAG GOENKA,THE ACHIEVERS SCHOOL,2270,8,Olympiad2020
4,ISHWARI BHANDE,RAHI PUBLIC SCHOOL,2661,8,Olympiad2020


In [161]:
Olympiad2021 = Olympiad2021[final_columns]
Olympiad2021['exam']='Olympiad2021'
Olympiad2021.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,AMBAR ARPITA GUPTA,DELHI PUBLIC SCHOOL,1332,10.0,Olympiad2021
1,SAANVI HITESH KHETAN,DELHI PUBLIC SCHOOL,1332,10.0,Olympiad2021
2,VAYUN AGRAWAL,DELHI PUBLIC SCHOOL,1332,10.0,Olympiad2021
3,DEVANSHI P LALWANI,PRIYADARSHINI NAGPUR PUBLIC SCHOOL,1064,10.0,Olympiad2021
4,JAY SUCHAK,PRIYADARSHINI NAGPUR PUBLIC SCHOOL,1064,10.0,Olympiad2021


In [165]:
OlympiadFeb23 = OlympiadFeb23[final_columns]
OlympiadFeb23['exam']='OlympiadFeb2023'
OlympiadFeb23.head(5)


,student_name,standard_school_name,standard_school_id,class,exam
1,AKSHAD R LANJEWAR,AADARSH TUTION CLASSES,2074,4.0,OlympiadFeb2023
2,MADHURA M SAWARKAR,AADARSH TUTION CLASSES,2074,4.0,OlympiadFeb2023
3,YUKTI M DURAGKAR,AADARSH TUTION CLASSES,2074,4.0,OlympiadFeb2023
4,HARSHALI KAWALE,AADARSH TUTION CLASSES,2074,4.0,OlympiadFeb2023
5,ARITA S FATEMA,AADARSH TUTION CLASSES,2074,4.0,OlympiadFeb2023


In [173]:
exam_cols = ['igko', 'ieo', 'nso', 'imo', 'nco', 'isso', 'iso']

OlympiadJune23['exam'] = OlympiadJune23[exam_cols].apply(
    lambda row: ', '.join([col for col in exam_cols if row[col] == 'Yes']),
    axis=1
)

OlympiadJune23.head()

,Unnamed: 0,sr_no,student_name,class,section,mobile_no,igko,ieo,nso,imo,nco,isso,iso,school,centre,year,standard_school_name,standard_school_addr,standard_school_city,standard_school_id,exam
0,0,1,ARCHANA ANIL VERMA,7,NaN,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,TIP TOP CONVENT SWAWALAMBI NAGAR,NaN,2023,TIP TOP CONVENT,SWAVALAMBI NAGAR NAGPUR 440022,NAGPUR,1770,imo
1,1,2,ADITY SHIVSHANKAR KANOJE,7,NaN,NaN,NaN,NaN,Yes,NaN,NaN,NaN,NaN,TIP TOP CONVENT SWAWALAMBI NAGAR,NaN,2023,TIP TOP CONVENT,SWAVALAMBI NAGAR NAGPUR 440022,NAGPUR,1770,nso
2,2,3,DEVSHREE VISHES TAJNE,7,NaN,NaN,Yes,NaN,NaN,Yes,NaN,NaN,NaN,TIP TOP CONVENT SWAWALAMBI NAGAR,NaN,2023,TIP TOP CONVENT,SWAVALAMBI NAGAR NAGPUR 440022,NAGPUR,1770,"igko, imo"
3,3,4,VIDISHA VAIBHAV KOYANDE,7,NaN,NaN,NaN,Yes,Yes,Yes,NaN,NaN,NaN,TIP TOP CONVENT SWAWALAMBI NAGAR,NaN,2023,TIP TOP CONVENT,SWAVALAMBI NAGAR NAGPUR 440022,NAGPUR,1770,"ieo, nso, imo"
4,4,5,SACHI SATISH BHAGAT,7,NaN,NaN,Yes,NaN,NaN,NaN,NaN,NaN,NaN,TIP TOP CONVENT SWAWALAMBI NAGAR,NaN,2023,TIP TOP CONVENT,SWAVALAMBI NAGAR NAGPUR 440022,NAGPUR,1770,igko


In [174]:
OlympiadJune23 = OlympiadJune23[final_columns+['exam']]
OlympiadJune23.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,ARCHANA ANIL VERMA,TIP TOP CONVENT,1770,7,imo
1,ADITY SHIVSHANKAR KANOJE,TIP TOP CONVENT,1770,7,nso
2,DEVSHREE VISHES TAJNE,TIP TOP CONVENT,1770,7,"igko, imo"
3,VIDISHA VAIBHAV KOYANDE,TIP TOP CONVENT,1770,7,"ieo, nso, imo"
4,SACHI SATISH BHAGAT,TIP TOP CONVENT,1770,7,igko


In [176]:
TalentData = TalentData[final_columns+['exam']]
TalentData.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,TASMAY MANOJ SHINGANE,PODAR INTERNATIONAL SCHOOL,1124,8,GANIT PRADNYA EXAMINATION 2023
1,ARYAN ASHISH HARKUT,MYRON ENGLISH MEDIUM SCHOOL,3262,8,GANIT PRADNYA EXAMINATION 2023
2,KRUSHNA SANTOSH GIRI,AVANT GRADE PUBLIC SCHOOL,1755,8,GANIT PRADNYA EXAMINATION 2023
3,KARTIK ASHISH BANG,NARAYANA VIDYALAYAM AMARAVATI,1410,8,GANIT PRADNYA EXAMINATION 2023
4,ABHIRAM ANAND DHOTE,PODAR INTERNATIONAL SCHOOL,1124,8,GANIT PRADNYA EXAMINATION 2023


In [190]:
VNITAxis23Mathamaze = VNITAxis23Mathamaze[final_columns]
VNITAxis23Mathamaze['exam']='VNITAxis23Mathamaze'
VNITAxis23Mathamaze.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,JIVTI KHURANA,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,2633,6.0,VNITAxis23Mathamaze
1,SWARA AMIT MUDALIAR,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR TRIMUTI NAGAR,1388,6.0,VNITAxis23Mathamaze
2,HAZEL PAHUJA,CENTRE POINT SCHOOL,1330,6.0,VNITAxis23Mathamaze
3,ADVIKA LOKHANDE,DELHI PUBLIC SCHOOL,1332,6.0,VNITAxis23Mathamaze
4,JIYANSH SUKHADIA,SFS NIMKHEDA,2029,6.0,VNITAxis23Mathamaze


In [189]:
VNITAxis23Mathemazeround1 = VNITAxis23Mathemazeround1[final_columns]
VNITAxis23Mathemazeround1['exam']='VNITAxis23Mathemazeround1'
VNITAxis23Mathemazeround1.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,AARNAV MANGE,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,2633,6.0,VNITAxis23Mathemazeround1
1,AKSHAJ RATHOD,SPM CONVENT,2285,6.0,VNITAxis23Mathemazeround1
2,ANIKA JAISWAL,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,2633,6.0,VNITAxis23Mathemazeround1
3,ANUJA MAHESH VARMA,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR TRIMUTI NAGAR,1388,6.0,VNITAxis23Mathemazeround1
4,ARNAV MAWLE,NARAYANA VIDYALAYAM,2629,6.0,VNITAxis23Mathemazeround1


In [188]:
VNITAxis23Mathemazeround2 = VNITAxis23Mathemazeround2[final_columns]
VNITAxis23Mathemazeround2['exam']='VNITAxis23Mathemazeround2'
VNITAxis23Mathemazeround2.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,AARNAV MANGE,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,2633,6.0,VNITAxis23Mathemazeround2
1,AKSHAJ RATHOD,SPM CONVENT,2285,6.0,VNITAxis23Mathemazeround2
2,ANIKA JAISWAL,BHAVANS BP VIDYA MANDIR SCHOOL CHINCHBHAVAN,2633,6.0,VNITAxis23Mathemazeround2
3,ANUJA MAHESH VARMA,BHAVANS BHAGWANDAS PUROHIT VIDYA MANDIR TRIMUTI NAGAR,1388,6.0,VNITAxis23Mathemazeround2
4,ARNAV MAWLE,NARAYANA VIDYALAYAM,2629,6.0,VNITAxis23Mathemazeround2


In [187]:
VNITAxis23Mathamaze['class'] = VNITAxis23Mathamaze['class'].astype(float) + 1
VNITAxis23Mathemazeround1['class'] = VNITAxis23Mathemazeround1['class'].astype(float) + 1
VNITAxis23Mathemazeround2['class'] = VNITAxis23Mathemazeround2['class'].astype(float) + 1

In [192]:
VNITAxis23exceptMathamaze['class'] = VNITAxis23exceptMathamaze['class'].astype(float) + 1
VNITAxis23exceptMathamaze = VNITAxis23exceptMathamaze[final_columns]
VNITAxis23exceptMathamaze['exam']='VNITAxis23exceptMathamaze'
VNITAxis23exceptMathamaze.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,AABHAS SUR,ST FRANCIS DE SALES HIGH SCHOOL,1367,6.0,VNITAxis23exceptMathamaze
1,AADYA SOURAV MULMELE,SOMALWAR NIKALAS HIGH SCHOOL,2636,6.0,VNITAxis23exceptMathamaze
2,ADHEESH MITHUN DESHMUKH,TIP TOP CONVENT,1432,6.0,VNITAxis23exceptMathamaze
3,ADHESH DESHMUKH,TIP TOP CONVENT,1432,6.0,VNITAxis23exceptMathamaze
4,ADITYA GUPTA,TIP TOP CONVENT,1432,6.0,VNITAxis23exceptMathamaze


In [194]:
leapData.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/leapData.csv', encoding='utf-8-sig')


Olympiad2019Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/Olympiad2019Ph2.csv', encoding='utf-8-sig')


Olympiad2020Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/Olympiad2020Ph2.csv', encoding='utf-8-sig')


Olympiad2021.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/Olympiad2021.csv', encoding='utf-8-sig')


OlympiadFeb23.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/OlympiadFeb23.csv', encoding='utf-8-sig')


OlympiadJune23.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/OlympiadJune23.csv', encoding='utf-8-sig')


TalentData.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/TalentData.csv', encoding='utf-8-sig')


# CDSData.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/CDSData.csv', encoding='utf-8-sig')


VNITAxis23Mathamaze.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/VNITAxis23Mathamaze.csv', encoding='utf-8-sig')


VNITAxis23Mathemazeround1.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/VNITAxis23Mathemazeround1.csv', encoding='utf-8-sig')


VNITAxis23Mathemazeround2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/VNITAxis23Mathemazeround2.csv', encoding='utf-8-sig')


VNITAxis23exceptMathamaze.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/VNITAxis23exceptMathamaze.csv', encoding='utf-8-sig')

In [203]:
Olympiad2019Ph2 = pd.read_excel('/Users/prathampandey/Downloads/data/1. 8, 9, 10, 11 Moving All Data.xlsx', 'Olympiad-2019 Ph-2')
Olympiad2019Ph2 = clean_column_names(Olympiad2019Ph2)
Olympiad2019Ph2.rename(columns={'name_of_student': 'student_name','school_name': 'school'}, inplace=True)
Olympiad2019Ph2['student_name'] = Olympiad2019Ph2['student_name'].apply(lambda x: clean_string_data(x))
Olympiad2019Ph2['school'] = Olympiad2019Ph2['school'].apply(lambda x: clean_string_data(x))
Olympiad2019Ph2['class'] = Olympiad2019Ph2['class'].apply(lambda x: str(x).replace('th', '')).astype(int)
Olympiad2019Ph2['standard_school_name'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[0])
Olympiad2019Ph2['standard_school_addr'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[1])
Olympiad2019Ph2['standard_school_city'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[2])
Olympiad2019Ph2['standard_school_id'] = Olympiad2019Ph2['school'].apply(lambda x: get_standard_school(x, school_master)[3])
Olympiad2019Ph2 = Olympiad2019Ph2[Olympiad2019Ph2['school'].notna() & (Olympiad2019Ph2['school'].str.strip() != '')]
Olympiad2019Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/sales/Olympiad2019Ph2_cleaned.csv', encoding='utf-8-sig')
Olympiad2019Ph2 = Olympiad2019Ph2[final_columns]
Olympiad2019Ph2['exam']='Olympiad2019'
Olympiad2019Ph2['class'] = Olympiad2019Ph2['class'].astype(float) + 5
Olympiad2019Ph2.to_csv('/Users/prathampandey/Documents/Code/Aks/icad_data_management/data/temp/Olympiad2019Ph2.csv', encoding='utf-8-sig')
Olympiad2019Ph2.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,VIDIT AJUNASH KANDEY,MOUNT LITERA ZEE SCHOOL,2552,14.0,Olympiad2019
1,RISHIKA DOBABRATA MUKHERJEE,BHAVANS LLOYDS VIDYA NIKETAN,1154,14.0,Olympiad2019
2,ADITYA ANIL VERMA,SACRED HEART ACADEMY,1358,14.0,Olympiad2019
3,SIDDHARTH ABHIVILAS NAKHATE,NARAYANA VIDYALAYAM AMARAVATI,1410,14.0,Olympiad2019
4,PRANJALI NAMDEV RATHOD,BHAVANS LLOYDS VIDYA NIKETAN,1154,14.0,Olympiad2019


In [218]:
VNITAxis23exceptMathamaze.count()

student_name            1259
standard_school_name    1259
standard_school_id      1259
class                   1259
exam                    1259
dtype: int64

In [219]:
dataframesMerge = [leapData,Olympiad2019Ph2,Olympiad2020Ph2,Olympiad2021,OlympiadFeb23,OlympiadJune23,TalentData,VNITAxis23Mathamaze,VNITAxis23Mathemazeround1,VNITAxis23Mathemazeround2,VNITAxis23exceptMathamaze]
combined_df = pd.concat(dataframesMerge, ignore_index=True)

In [220]:
combined_df.count()

student_name            49045
standard_school_name    49045
standard_school_id      49045
class                   49045
exam                    49045
dtype: int64

In [221]:
combined_df.head(5)

,student_name,standard_school_name,standard_school_id,class,exam
0,SIDHANT BOBADE,DNYANMATA HIGH SCHOOL AMRAVATI,1111,7.0,LEAP
1,PARTH BONDE,KK CAMBRIDGE SCHOOL,2162,7.0,LEAP
2,NINAD MATKAR,ORCHID CITY INTERNATIONAL SCHOOL AMRAVATI,2232,7.0,LEAP
3,RACHIT RAVI TAYADE,DNYANMATA HIGH SCHOOL AMRAVATI,1111,7.0,LEAP
4,PRANJAL P KANTUTE,KK CAMBRIDGE SCHOOL,2162,7.0,LEAP


In [222]:
agg_df = combined_df.groupby(
    ['student_name', 'standard_school_name'], dropna=False
).agg({
    'standard_school_id': 'first',  # or use set if IDs differ
    'class': lambda x: ', '.join(sorted(set(str(v) for v in x if pd.notnull(v)))),
    'exam': lambda x: ', '.join(sorted(set(e.strip() for e in ', '.join(x.dropna()).split(','))))
}).reset_index()

In [223]:
agg_df.count()

student_name            46770
standard_school_name    46770
standard_school_id      46770
class                   46770
exam                    46770
dtype: int64

In [225]:
agg_df.value_counts('class')

class
10.0                22928
9.0                  9120
8.0                  6354
7.0                  6124
11.0                  525
6.0                   392
4.0                   358
5.0                   354
12.0                  243
15.0                   89
13.0                   71
10.0, 9.0              53
10.0, 11.0             34
8.0, 9.0               24
7.0, 8.0               17
11.0, 8.0              11
12.0, 9.0              10
14.0                    9
10.0, 7.0               8
10.0, 8.0               6
11.0, 12.0              6
7.0, 9.0                4
12.0, 8.0               3
13.0, 9.0               3
10.0, 12.0              2
13.0, 8.0               2
11.0, 9.0               2
6.0, 9.0                2
11.0, 15.0              2
10.0, 15.0              2
5.0, 6.0                2
12.0, 13.0              2
1.0                     1
6.0, 7.0                1
1.0, 4.0                1
10.0, 7.0, 8.0          1
10.0, 11.0, 9.0         1
10.0, 11.0, 7.0         1
10.0, 

In [226]:
admissionData = pd.read_excel('/Users/prathampandey/Downloads/data/1. Admission - 2026 Batch.xlsx')

In [231]:
admissionData.head(5)

,sr_no,roll_no,student_name,on_roll,branch,batch,batch_id,on_roll1,parent_no,communication_no,student_phone_no,group,category,group_id,caste_category,cpa,cpa_date,software_id,cpa_roll_no,date_of_adm,school,board_in_10th,medium_of_examination,board_in_11th_std,xi_college,stream,bif_batch,phone,suspension_date,dropout_date,remark,reshuffledate,phase,old_phase,student_information_update_jan21,batch_id_old,classes_start_date,classes_on_turns,target_group_0,batch_start,city,email,true
0,1,260111101,Sejal Nikhil Meshram,NaN,HO,zExcel-JEE26-SUPER ALPHA-3,JEE26 P1,Dropout,NaN,NaN,NaN,NaN,Regular batch,NaN,SC,79.20,NaN,NaN,NaN,NaT,"Tip Top Convent, Swavalambi Nagar, Nagpur",STATE,NaN,State Board,TIP TOP CONVENT,NaN,Information Technology,NaN,NaT,2024-09-11,FATHER INFORMED TO WITHDROW ADMISSION RESON NOT SPECIFIED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,260111102,Anushka Kiran Mankar,NaN,HO,Excel-JEE26-SUPER ALPHA-3,JEE26 P1,On Roll,NaN,NaN,NaN,NaN,Regular batch,NaN,NaN,79.20,NaN,NaN,NaN,NaT,"Tip Top Convent, Swavalambi Nagar, Nagpur",STATE,NaN,State Board,M.K UMATE,NaN,Electronics,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,260111103,Samruddhi Pravin Nitnaware,NaN,HO,Excel-JEE26-SUPER ALPHA-2,JEE26 P1,On Roll,NaN,NaN,NaN,NaN,Regular batch,NaN,NaN,75.00,NaN,NaN,NaN,NaT,"Tip Top Convent, Swavalambi Nagar, Nagpur",STATE,NaN,State Board,VIDYA SADHANA,NaN,Computer Science,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,260111104,Saksham Vikas Khobragade,NaN,HO,Excel-JEE26-SUPER ALPHA-1,JEE26 P1,On Roll,NaN,NaN,NaN,NaN,Regular batch,NaN,OBC,70.33,NaN,NaN,NaN,NaT,"St. Johns Public School ( CBSE), Mohan Nagar, Nagpur",CBSE,English,State Board,ST.PAUL COLLEGE,NaN,Computer Science,NaN,NaT,NaT,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,260111105,Sneha Sudhir Panchware,NaN,HO,Excel-JEE26-SUPER ALPHA-1,JEE26 P1,On Roll,NaN,NaN,NaN,NaN,Regular batch,NaN,NaN,70.33,NaN,NaN,NaN,NaT,"Narayana Vidyalayam, Wardha Road, Nagpur",CBSE,NaN,CBSE,NARAYANA VDYALAYAM,NaN,Physical Education,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [228]:
admissionData = clean_column_names(admissionData)

In [230]:
admissionData.rename(columns={'students_name': 'student_name','name_of_school_10th': 'school'}, inplace=True)

In [232]:
final_columns = ['student_name', 'branch', 'batch', 'on_roll1','category','school','xi_college','board_in_10th','board_in_11th_std',]

In [233]:
admissionData = admissionData[final_columns] 

In [234]:
admissionData.head(5)

,student_name,branch,batch,on_roll1,category,school,xi_college,board_in_10th,board_in_11th_std
0,Sejal Nikhil Meshram,HO,zExcel-JEE26-SUPER ALPHA-3,Dropout,Regular batch,"Tip Top Convent, Swavalambi Nagar, Nagpur",TIP TOP CONVENT,STATE,State Board
1,Anushka Kiran Mankar,HO,Excel-JEE26-SUPER ALPHA-3,On Roll,Regular batch,"Tip Top Convent, Swavalambi Nagar, Nagpur",M.K UMATE,STATE,State Board
2,Samruddhi Pravin Nitnaware,HO,Excel-JEE26-SUPER ALPHA-2,On Roll,Regular batch,"Tip Top Convent, Swavalambi Nagar, Nagpur",VIDYA SADHANA,STATE,State Board
3,Saksham Vikas Khobragade,HO,Excel-JEE26-SUPER ALPHA-1,On Roll,Regular batch,"St. Johns Public School ( CBSE), Mohan Nagar, Nagpur",ST.PAUL COLLEGE,CBSE,State Board
4,Sneha Sudhir Panchware,HO,Excel-JEE26-SUPER ALPHA-1,On Roll,Regular batch,"Narayana Vidyalayam, Wardha Road, Nagpur",NARAYANA VDYALAYAM,CBSE,CBSE


In [235]:
admissionData['student_name'] = admissionData['student_name'].apply(lambda x: clean_string_data(x))
admissionData['school'] = admissionData['school'].apply(lambda x: clean_string_data(x))
admissionData['branch'] = admissionData['branch'].apply(lambda x: clean_string_data(x))
admissionData['batch'] = admissionData['batch'].apply(lambda x: clean_string_data(x))
admissionData['on_roll1'] = admissionData['on_roll1'].apply(lambda x: clean_string_data(x))
admissionData['category'] = admissionData['category'].apply(lambda x: clean_string_data(x))
admissionData['xi_college'] = admissionData['xi_college'].apply(lambda x: clean_string_data(x))
admissionData['board_in_10th'] = admissionData['board_in_10th'].apply(lambda x: clean_string_data(x))
admissionData['board_in_11th_std'] = admissionData['board_in_11th_std'].apply(lambda x: clean_string_data(x))